This is a quite deep neural network with multiple convolution blocks. Each of these blocks narrows the image into a set of pooled features before running through two large dense layers to classify the final images. 

In [1]:
import keras
from keras.datasets import mnist
from keras.models import Sequential,Model
from keras.layers import Dense, Dropout,Input,Flatten,MaxPooling2D,Conv2D,MaxPooling1D,AlphaDropout
import numpy as np

This bit of code is a quick way to tell if you are running with a GPU available. For a large, deep network such as this -- working with a GPU can be 20-50x faster. 

In [2]:
from tensorflow.python.client import device_lib
print(device_lib.list_local_devices())

[name: "/device:CPU:0"
device_type: "CPU"
memory_limit: 268435456
locality {
}
incarnation: 4098663538251613870
xla_global_id: -1
, name: "/device:GPU:0"
device_type: "GPU"
locality {
  bus_id: 1
}
incarnation: 11104258365552681595
physical_device_desc: "device: 0, name: METAL, pci bus id: <undefined>"
xla_global_id: -1
]


2026-08-16 00:11:46.546218: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-08-16 00:11:46.546242: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-08-16 00:11:46.546246: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.00 GB
I0000 00:00:1786819306.546261 1782732 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1786819306.546281 1782732 pluggable_device_factory.cc:271] Created TensorFlow device (/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [3]:
(x_train,y_train),(x_test,y_test)=mnist.load_data()
x_train = np.expand_dims(x_train/np.max(x_train), axis= -1)
x_test = np.expand_dims(x_test/np.max(x_test),axis=-1)
y_train = keras.utils.to_categorical(y_train, 10)
y_test = keras.utils.to_categorical(y_test,10)


In [4]:
input_shape = x_train[0].shape
classes = 10
inputs = Input(shape=input_shape)

#Block 1
x = Conv2D(64,(3,3),activation='relu',padding="same",name="block1_conv_1")(inputs)
x = Conv2D(64,(3,3),activation='relu',padding="same",name="block1_conv_2")(x)
x = MaxPooling2D((2,2),strides=(2,2),name="block1_pool")(x)

#Block 2
x = Conv2D(128,(3,3),activation='relu',padding="same",name="block2_conv_1")(x)
x = Conv2D(128,(3,3),activation='relu',padding="same",name="block2_conv_2")(x)
x = MaxPooling2D((2,2), strides=(2,2), name="block2_pool")(x)

#Block 3
x = Conv2D(256,(3,3),activation='relu',padding="same",name="block3_conv_1")(x)
x = Conv2D(256,(3,3),activation='relu',padding="same",name="block3_conv_2")(x)
x = Conv2D(256,(3,3),activation='relu',padding="same",name="block3_conv_3")(x)
x = MaxPooling2D((2,2), strides=(2,2), name="block3_pool")(x)

#Classification Block
x = Flatten(name="flatten")(x)
x = Dense(512, activation='relu', name="fc1")(x)
x = Dense(512,activation='relu',name="fc2")(x)
outputs = Dense(classes, activation='softmax',name="predictions")(x)

model = Model(inputs=inputs, outputs=outputs)
model.summary()

I0000 00:00:1786819306.658490 1782732 pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
I0000 00:00:1786819306.658516 1782732 pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv_1 (Conv2D)          │ (None, 28, 28, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv_2 (Conv2D)          │ (None, 28, 28, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv_1 (Conv2D)          │ (None, 14, 14, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv_2 (Conv2D)          │ (None, 14, 14, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv_1 (Conv2D)          │ (None, 7, 7, 256)      │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv_2 (Conv2D)          │ (None, 7, 7, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv_3 (Conv2D)          │ (None, 7, 7, 256)      │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 3, 3, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2304)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc1 (Dense)                     │ (None, 512)            │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ fc2 (Dense)                     │ (None, 512)            │       262,656 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 10)             │         5,130 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,182,282 (12.14 MB)

 Trainable params: 3,182,282 (12.14 MB)

 Non-trainable params: 0 (0.00 B)

With the model assembled, we compile it, which prepares the model for execution with a solver. And fit to the training data. This is similar to what we did with the Classical/Dense network -- in fact identical. 

In [5]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
history= model.fit(x_train,y_train,batch_size=64,epochs=8,verbose=1,validation_data=(x_test,y_test))


Epoch 1/8


2026-08-16 00:11:47.296074: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


938/938 ━━━━━━━━━━━━━━━━━━━━ 38s 39ms/step - accuracy: 0.9395 - loss: 0.1991 - val_accuracy: 0.9617 - val_loss: 0.1782
Epoch 2/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 36s 39ms/step - accuracy: 0.9616 - loss: 2.8562 - val_accuracy: 0.9837 - val_loss: 0.0496
Epoch 3/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 53s 56ms/step - accuracy: 0.9869 - loss: 0.0460 - val_accuracy: 0.9869 - val_loss: 0.0454
Epoch 4/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 51s 55ms/step - accuracy: 0.9877 - loss: 0.0455 - val_accuracy: 0.9880 - val_loss: 0.0423
Epoch 5/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 50s 53ms/step - accuracy: 0.9890 - loss: 0.0444 - val_accuracy: 0.9842 - val_loss: 0.0603
Epoch 6/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 49s 52ms/step - accuracy: 0.9662 - loss: 1409.8192 - val_accuracy: 0.9815 - val_loss: 75.7186
Epoch 7/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 51s 55ms/step - accuracy: 0.9808 - loss: 61.9091 - val_accuracy: 0.9831 - val_loss: 34.7841
Epoch 8/8
938/938 ━━━━━━━━━━━━━━━━━━━━ 50s 53ms/step - accuracy: 0.9801 - loss: 77.8673 - val_accurac

Now, accuracy got improved. You can experiment with different hyper parameters. 